# Data Mining Assignment - Data Lake Preprocessing
**Name:** Safiya Nasir
**Roll Number:** LCI2023053

In [ ]:
!pip install pyspark

### 1. Data Lake Structure
Creating the bronze and silver folders.

In [ ]:
import os
import shutil

os.makedirs('data_lake/bronze', exist_ok=True)
os.makedirs('data_lake/silver/preprocessed_sales', exist_ok=True)

# Assuming sales.csv is uploaded in Colab in the current directory
if os.path.exists('sales.csv'):
    shutil.copy('sales.csv', 'data_lake/bronze/sales.csv')

### 2. Load and Inspect with Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, when, to_date, count, isnan, abs

spark = SparkSession.builder \
    .appName("Data Lake Preprocessing") \
    .getOrCreate()

bronze_path = "data_lake/bronze/sales.csv"
# Note: If running locally without 'sales.csv' in colab, please upload sales.csv first.
# For this assignment, we assume it's loaded to data_lake/bronze/sales.csv
df = spark.read.csv(bronze_path, header=True, inferSchema=True)

print("--- Schema ---")
df.printSchema()

print("\n--- First 5 rows ---")
df.show(5)

print(f"\nTotal record count: {df.count()}")

In [ ]:
print("--- Data Quality Issues ---")
# Duplicates
duplicates = df.count() - df.dropDuplicates().count()
print(f"Exact duplicate rows: {duplicates}")

# Nulls
print("\nNull values per column:")
df.select([count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) for c in df.columns]).show()


### 3. Basic Preprocessing - Silver Layer

In [ ]:
# 1. Remove exact duplicate rows
df_silver = df.dropDuplicates()

# 2. Handle missing values (fillna with 'Unknown' for text, 0 for numeric)
df_silver = df_silver.fillna('Unknown')
df_silver = df_silver.fillna(0)

# 3. Trim extra spaces and standardize capitalization
text_columns = ['customer_name', 'product', 'category', 'payment_method', 'city', 'state', 'order_status']
for c in text_columns:
    df_silver = df_silver.withColumn(c, trim(initcap(col(c))))

# 4. Apply basic validation to quantity, unit price, and discount percentage
df_silver = df_silver.withColumn("quantity", abs(col("quantity")))
df_silver = df_silver.withColumn("unit_price", abs(col("unit_price")))
df_silver = df_silver.withColumn("discount_percent", 
                                 when(col("discount_percent") < 0, 0)
                                 .when(col("discount_percent") > 100, 100)
                                 .otherwise(col("discount_percent")))

# 5. Convert valid dates to one consistent format
df_silver = df_silver.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

# Save to silver layer
silver_path = "data_lake/silver/preprocessed_sales"
df_silver.write.mode("overwrite").csv(silver_path, header=True)
print("Data successfully saved to Silver layer in CSV format.")

### 4. Verification

In [ ]:
print(f"Row count before preprocessing: {df.count()}")
print(f"Row count after preprocessing: {df_silver.count()}")

print("\n--- Sample of Preprocessed Silver Data ---")
df_silver_read = spark.read.csv(silver_path, header=True, inferSchema=True)
df_silver_read.show(5)

#### Preprocessing Operations Performed:
1. **Duplicates:** Removed all exact duplicate rows.
2. **Null Values:** Handled missing string values by substituting 'Unknown' and missing numerical values with 0.
3. **Text Formatting:** Standardized capitalization (using `initcap`) and removed trailing/leading spaces for all text fields.
4. **Validation:** Applied absolute conversion to `quantity` and `unit_price` to handle negatives. Constrained `discount_percent` to a valid 0-100 range.
5. **Dates:** Formatted the `order_date` field uniformly to `yyyy-MM-dd`.